# ZGLMM368 — Centro 4014 — Datalake

**Tabela:** `dev_procurement.corp_curated.tbl_ds_pro_zglmm368`
**Domínio:** Reservas de material
**Filtro do cenário:** `cod_centro = '4014'`
**Colunas:** 36 · **Clustering declarado:** `num_reserva`, `cod_material`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente com o filtro do centro embutido. Não há widget, view temporária nem ordem obrigatória.

## Objetivo

Extrair e caracterizar **toda** a base do centro 4014 para comparação com o extrato do SAP.

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados da tabela |
| 2 | Volumetria e representatividade do cenário |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| 6 | Preenchimento de todas as colunas |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 9 | Perfil numérico |
| **10** | **Totais para conciliação com o SAP** |
| 11 | Datas |
| 12 | Códigos e zeros à esquerda |
| **13** | **Chaves normalizadas para join** |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| 18 | Análises específicas |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo do cenário |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados da tabela

In [ ]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_pro_zglmm368;

In [ ]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_pro_zglmm368;

In [ ]:
-- Ultimas gravacoes (falha se for view)
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_zglmm368 LIMIT 20;

## 2. Volumetria e representatividade

Quanto o centro 4014 representa do total da tabela.

In [ ]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368)                                   AS linhas_tabela_toda,
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014')                               AS linhas_centro_4014,
  ROUND(100.0 * (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014')
              / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368), 4)                  AS pct_do_total,
  (SELECT COUNT(DISTINCT `cod_centro`) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368)                     AS centros_na_tabela;

## 3. Confirmação do filtro

Confirma que o valor `4014` existe e que não há variação de formato
(espaços, zeros à esquerda) que faça o filtro perder linhas silenciosamente.

**Se retornar mais de uma linha, o filtro `= '4014'` está incompleto.**

In [ ]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(`cod_centro` AS STRING)                    AS valor_bruto,
       length(CAST(`cod_centro` AS STRING))            AS comprimento,
       COUNT(*)                                   AS linhas
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368
WHERE regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') = '4014'
   OR trim(CAST(`cod_centro` AS STRING)) = '4014'
GROUP BY CAST(`cod_centro` AS STRING), length(CAST(`cod_centro` AS STRING))
ORDER BY linhas DESC;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [ ]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'),
g AS (
  SELECT 'num_reserva' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_reserva + cod_material' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva`, `cod_material` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_reserva + num_item_reserva' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_reserva + num_item_reserva + cod_material' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `cod_material` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014')
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `num_reserva + cod_material`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [ ]:
-- 5. CHAVES DUPLICADAS
SELECT `num_reserva`, `cod_material`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
GROUP BY `num_reserva`, `cod_material`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [ ]:
-- 5.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
),
dup AS (
  SELECT `num_reserva`, `cod_material` FROM cen GROUP BY `num_reserva`, `cod_material` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`num_reserva` <=> dup.`num_reserva` AND c.`cod_material` <=> dup.`cod_material`
),
agg AS (
  SELECT `num_reserva`, `cod_material`,
         COUNT(DISTINCT `cod_empresa_compensacao`) AS `cod_empresa_compensacao`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `cod_deposito`) AS `cod_deposito`,
         COUNT(DISTINCT `num_ordem`) AS `num_ordem`,
         COUNT(DISTINCT `desc_produto`) AS `desc_produto`,
         COUNT(DISTINCT `vl_qtd_solicitada`) AS `vl_qtd_solicitada`,
         COUNT(DISTINCT `vl_qtd_retirada`) AS `vl_qtd_retirada`,
         COUNT(DISTINCT `sg_unidade_medida_basica`) AS `sg_unidade_medida_basica`,
         COUNT(DISTINCT `qt_estoque_livre_avaliado`) AS `qt_estoque_livre_avaliado`,
         COUNT(DISTINCT `num_item_reserva`) AS `num_item_reserva`,
         COUNT(DISTINCT `nm_usuario`) AS `nm_usuario`,
         COUNT(DISTINCT `dt_reserva`) AS `dt_reserva`,
         COUNT(DISTINCT `cod_tipo_movimento`) AS `cod_tipo_movimento`,
         COUNT(DISTINCT `dt_necessidade`) AS `dt_necessidade`,
         COUNT(DISTINCT `num_lote`) AS `num_lote`,
         COUNT(DISTINCT `qt_estoque_consignado`) AS `qt_estoque_consignado`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `nm_ponto_descarga`) AS `nm_ponto_descarga`,
         COUNT(DISTINCT `nm_recebedor_mercadoria`) AS `nm_recebedor_mercadoria`,
         COUNT(DISTINCT `ind_registro_final`) AS `ind_registro_final`,
         COUNT(DISTINCT `ind_item_eliminado`) AS `ind_item_eliminado`,
         COUNT(DISTINCT `ind_movimento_permitido`) AS `ind_movimento_permitido`,
         COUNT(DISTINCT `num_imobilizado`) AS `num_imobilizado`,
         COUNT(DISTINCT `ind_item_dummy`) AS `ind_item_dummy`,
         COUNT(DISTINCT `ind_material_granel`) AS `ind_material_granel`,
         COUNT(DISTINCT `num_sub_imobilizado`) AS `num_sub_imobilizado`,
         COUNT(DISTINCT `st_reserva`) AS `st_reserva`,
         COUNT(DISTINCT `tp_registro`) AS `tp_registro`,
         COUNT(DISTINCT `cod_centro_custo`) AS `cod_centro_custo`,
         COUNT(DISTINCT `cod_deposito_destino`) AS `cod_deposito_destino`,
         COUNT(DISTINCT `cod_diagrama_rede`) AS `cod_diagrama_rede`,
         COUNT(DISTINCT `num_divisao_programa_venda`) AS `num_divisao_programa_venda`,
         COUNT(DISTINCT `ind_necessidade_atendida`) AS `ind_necessidade_atendida`,
         COUNT(DISTINCT `qt_estoque_projeto`) AS `qt_estoque_projeto`
  FROM d GROUP BY `num_reserva`, `cod_material`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(34,
    'cod_empresa_compensacao', MAX(`cod_empresa_compensacao`),
    'cod_centro', MAX(`cod_centro`),
    'cod_deposito', MAX(`cod_deposito`),
    'num_ordem', MAX(`num_ordem`),
    'desc_produto', MAX(`desc_produto`),
    'vl_qtd_solicitada', MAX(`vl_qtd_solicitada`),
    'vl_qtd_retirada', MAX(`vl_qtd_retirada`),
    'sg_unidade_medida_basica', MAX(`sg_unidade_medida_basica`),
    'qt_estoque_livre_avaliado', MAX(`qt_estoque_livre_avaliado`),
    'num_item_reserva', MAX(`num_item_reserva`),
    'nm_usuario', MAX(`nm_usuario`),
    'dt_reserva', MAX(`dt_reserva`),
    'cod_tipo_movimento', MAX(`cod_tipo_movimento`),
    'dt_necessidade', MAX(`dt_necessidade`),
    'num_lote', MAX(`num_lote`),
    'qt_estoque_consignado', MAX(`qt_estoque_consignado`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'nm_ponto_descarga', MAX(`nm_ponto_descarga`),
    'nm_recebedor_mercadoria', MAX(`nm_recebedor_mercadoria`),
    'ind_registro_final', MAX(`ind_registro_final`),
    'ind_item_eliminado', MAX(`ind_item_eliminado`),
    'ind_movimento_permitido', MAX(`ind_movimento_permitido`),
    'num_imobilizado', MAX(`num_imobilizado`),
    'ind_item_dummy', MAX(`ind_item_dummy`),
    'ind_material_granel', MAX(`ind_material_granel`),
    'num_sub_imobilizado', MAX(`num_sub_imobilizado`),
    'st_reserva', MAX(`st_reserva`),
    'tp_registro', MAX(`tp_registro`),
    'cod_centro_custo', MAX(`cod_centro_custo`),
    'cod_deposito_destino', MAX(`cod_deposito_destino`),
    'cod_diagrama_rede', MAX(`cod_diagrama_rede`),
    'num_divisao_programa_venda', MAX(`num_divisao_programa_venda`),
    'ind_necessidade_atendida', MAX(`ind_necessidade_atendida`),
    'qt_estoque_projeto', MAX(`qt_estoque_projeto`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada **neste centro**.

Uma coluna pode ter dado na tabela toda e estar vazia no centro 4014 — ou o contrário.
Por isso a varredura é feita sobre o recorte, não sobre a base completa.

In [ ]:
-- 6. PREENCHIMENTO NO CENTRO 4014
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'),
perf AS (
  SELECT stack(36,
    'cod_empresa_compensacao', 'string', COUNT_IF(`cod_empresa_compensacao` IS NULL), COUNT_IF(`cod_empresa_compensacao` IS NOT NULL AND lower(trim(`cod_empresa_compensacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa_compensacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'num_reserva', 'string', COUNT_IF(`num_reserva` IS NULL), COUNT_IF(`num_reserva` IS NOT NULL AND lower(trim(`num_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_reserva`) RLIKE '^0+([.,]0+)?$'),
    'num_ordem', 'string', COUNT_IF(`num_ordem` IS NULL), COUNT_IF(`num_ordem` IS NOT NULL AND lower(trim(`num_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_produto', 'string', COUNT_IF(`desc_produto` IS NULL), COUNT_IF(`desc_produto` IS NOT NULL AND lower(trim(`desc_produto`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_produto`) RLIKE '^0+([.,]0+)?$'),
    'vl_qtd_solicitada', 'decimal(13,3)', COUNT_IF(`vl_qtd_solicitada` IS NULL), 0L, COUNT_IF(`vl_qtd_solicitada` = 0),
    'vl_qtd_retirada', 'decimal(13,3)', COUNT_IF(`vl_qtd_retirada` IS NULL), 0L, COUNT_IF(`vl_qtd_retirada` = 0),
    'sg_unidade_medida_basica', 'string', COUNT_IF(`sg_unidade_medida_basica` IS NULL), COUNT_IF(`sg_unidade_medida_basica` IS NOT NULL AND lower(trim(`sg_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$'),
    'qt_estoque_livre_avaliado', 'decimal(13,3)', COUNT_IF(`qt_estoque_livre_avaliado` IS NULL), 0L, COUNT_IF(`qt_estoque_livre_avaliado` = 0),
    'num_item_reserva', 'string', COUNT_IF(`num_item_reserva` IS NULL), COUNT_IF(`num_item_reserva` IS NOT NULL AND lower(trim(`num_item_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_reserva`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario', 'string', COUNT_IF(`nm_usuario` IS NULL), COUNT_IF(`nm_usuario` IS NOT NULL AND lower(trim(`nm_usuario`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario`) RLIKE '^0+([.,]0+)?$'),
    'dt_reserva', 'string', COUNT_IF(`dt_reserva` IS NULL), COUNT_IF(`dt_reserva` IS NOT NULL AND lower(trim(`dt_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_reserva`) RLIKE '^0+([.,]0+)?$'),
    'cod_tipo_movimento', 'string', COUNT_IF(`cod_tipo_movimento` IS NULL), COUNT_IF(`cod_tipo_movimento` IS NOT NULL AND lower(trim(`cod_tipo_movimento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_tipo_movimento`) RLIKE '^0+([.,]0+)?$'),
    'dt_necessidade', 'date', COUNT_IF(`dt_necessidade` IS NULL), 0L, 0L,
    'num_lote', 'string', COUNT_IF(`num_lote` IS NULL), COUNT_IF(`num_lote` IS NOT NULL AND lower(trim(`num_lote`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_lote`) RLIKE '^0+([.,]0+)?$'),
    'qt_estoque_consignado', 'decimal(23,3)', COUNT_IF(`qt_estoque_consignado` IS NULL), 0L, COUNT_IF(`qt_estoque_consignado` = 0),
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'nm_ponto_descarga', 'string', COUNT_IF(`nm_ponto_descarga` IS NULL), COUNT_IF(`nm_ponto_descarga` IS NOT NULL AND lower(trim(`nm_ponto_descarga`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_ponto_descarga`) RLIKE '^0+([.,]0+)?$'),
    'nm_recebedor_mercadoria', 'string', COUNT_IF(`nm_recebedor_mercadoria` IS NULL), COUNT_IF(`nm_recebedor_mercadoria` IS NOT NULL AND lower(trim(`nm_recebedor_mercadoria`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_recebedor_mercadoria`) RLIKE '^0+([.,]0+)?$'),
    'ind_registro_final', 'string', COUNT_IF(`ind_registro_final` IS NULL), COUNT_IF(`ind_registro_final` IS NOT NULL AND lower(trim(`ind_registro_final`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_registro_final`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_eliminado', 'string', COUNT_IF(`ind_item_eliminado` IS NULL), COUNT_IF(`ind_item_eliminado` IS NOT NULL AND lower(trim(`ind_item_eliminado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_item_eliminado`) RLIKE '^0+([.,]0+)?$'),
    'ind_movimento_permitido', 'string', COUNT_IF(`ind_movimento_permitido` IS NULL), COUNT_IF(`ind_movimento_permitido` IS NOT NULL AND lower(trim(`ind_movimento_permitido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_movimento_permitido`) RLIKE '^0+([.,]0+)?$'),
    'num_imobilizado', 'string', COUNT_IF(`num_imobilizado` IS NULL), COUNT_IF(`num_imobilizado` IS NOT NULL AND lower(trim(`num_imobilizado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_imobilizado`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_dummy', 'string', COUNT_IF(`ind_item_dummy` IS NULL), COUNT_IF(`ind_item_dummy` IS NOT NULL AND lower(trim(`ind_item_dummy`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_item_dummy`) RLIKE '^0+([.,]0+)?$'),
    'ind_material_granel', 'string', COUNT_IF(`ind_material_granel` IS NULL), COUNT_IF(`ind_material_granel` IS NOT NULL AND lower(trim(`ind_material_granel`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_material_granel`) RLIKE '^0+([.,]0+)?$'),
    'num_sub_imobilizado', 'string', COUNT_IF(`num_sub_imobilizado` IS NULL), COUNT_IF(`num_sub_imobilizado` IS NOT NULL AND lower(trim(`num_sub_imobilizado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_sub_imobilizado`) RLIKE '^0+([.,]0+)?$'),
    'st_reserva', 'string', COUNT_IF(`st_reserva` IS NULL), COUNT_IF(`st_reserva` IS NOT NULL AND lower(trim(`st_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_reserva`) RLIKE '^0+([.,]0+)?$'),
    'tp_registro', 'string', COUNT_IF(`tp_registro` IS NULL), COUNT_IF(`tp_registro` IS NOT NULL AND lower(trim(`tp_registro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_registro`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo', 'string', COUNT_IF(`cod_centro_custo` IS NULL), COUNT_IF(`cod_centro_custo` IS NOT NULL AND lower(trim(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito_destino', 'string', COUNT_IF(`cod_deposito_destino` IS NULL), COUNT_IF(`cod_deposito_destino` IS NOT NULL AND lower(trim(`cod_deposito_destino`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito_destino`) RLIKE '^0+([.,]0+)?$'),
    'cod_diagrama_rede', 'string', COUNT_IF(`cod_diagrama_rede` IS NULL), COUNT_IF(`cod_diagrama_rede` IS NOT NULL AND lower(trim(`cod_diagrama_rede`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_diagrama_rede`) RLIKE '^0+([.,]0+)?$'),
    'num_divisao_programa_venda', 'string', COUNT_IF(`num_divisao_programa_venda` IS NULL), COUNT_IF(`num_divisao_programa_venda` IS NOT NULL AND lower(trim(`num_divisao_programa_venda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_divisao_programa_venda`) RLIKE '^0+([.,]0+)?$'),
    'ind_necessidade_atendida', 'string', COUNT_IF(`ind_necessidade_atendida` IS NULL), COUNT_IF(`ind_necessidade_atendida` IS NOT NULL AND lower(trim(`ind_necessidade_atendida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_necessidade_atendida`) RLIKE '^0+([.,]0+)?$'),
    'qt_estoque_projeto', 'decimal(23,3)', COUNT_IF(`qt_estoque_projeto` IS NULL), 0L, COUNT_IF(`qt_estoque_projeto` = 0)
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                       THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                              AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade no cenário

In [ ]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'),
card AS (
  SELECT stack(36,
    'cod_empresa_compensacao', 'string', approx_count_distinct(`cod_empresa_compensacao`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_deposito', 'string', approx_count_distinct(`cod_deposito`),
    'num_reserva', 'string', approx_count_distinct(`num_reserva`),
    'num_ordem', 'string', approx_count_distinct(`num_ordem`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_produto', 'string', approx_count_distinct(`desc_produto`),
    'vl_qtd_solicitada', 'decimal(13,3)', approx_count_distinct(`vl_qtd_solicitada`),
    'vl_qtd_retirada', 'decimal(13,3)', approx_count_distinct(`vl_qtd_retirada`),
    'sg_unidade_medida_basica', 'string', approx_count_distinct(`sg_unidade_medida_basica`),
    'qt_estoque_livre_avaliado', 'decimal(13,3)', approx_count_distinct(`qt_estoque_livre_avaliado`),
    'num_item_reserva', 'string', approx_count_distinct(`num_item_reserva`),
    'nm_usuario', 'string', approx_count_distinct(`nm_usuario`),
    'dt_reserva', 'string', approx_count_distinct(`dt_reserva`),
    'cod_tipo_movimento', 'string', approx_count_distinct(`cod_tipo_movimento`),
    'dt_necessidade', 'date', approx_count_distinct(`dt_necessidade`),
    'num_lote', 'string', approx_count_distinct(`num_lote`),
    'qt_estoque_consignado', 'decimal(23,3)', approx_count_distinct(`qt_estoque_consignado`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'nm_ponto_descarga', 'string', approx_count_distinct(`nm_ponto_descarga`),
    'nm_recebedor_mercadoria', 'string', approx_count_distinct(`nm_recebedor_mercadoria`),
    'ind_registro_final', 'string', approx_count_distinct(`ind_registro_final`),
    'ind_item_eliminado', 'string', approx_count_distinct(`ind_item_eliminado`),
    'ind_movimento_permitido', 'string', approx_count_distinct(`ind_movimento_permitido`),
    'num_imobilizado', 'string', approx_count_distinct(`num_imobilizado`),
    'ind_item_dummy', 'string', approx_count_distinct(`ind_item_dummy`),
    'ind_material_granel', 'string', approx_count_distinct(`ind_material_granel`),
    'num_sub_imobilizado', 'string', approx_count_distinct(`num_sub_imobilizado`),
    'st_reserva', 'string', approx_count_distinct(`st_reserva`),
    'tp_registro', 'string', approx_count_distinct(`tp_registro`),
    'cod_centro_custo', 'string', approx_count_distinct(`cod_centro_custo`),
    'cod_deposito_destino', 'string', approx_count_distinct(`cod_deposito_destino`),
    'cod_diagrama_rede', 'string', approx_count_distinct(`cod_diagrama_rede`),
    'num_divisao_programa_venda', 'string', approx_count_distinct(`num_divisao_programa_venda`),
    'ind_necessidade_atendida', 'string', approx_count_distinct(`ind_necessidade_atendida`),
    'qt_estoque_projeto', 'decimal(23,3)', approx_count_distinct(`qt_estoque_projeto`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma, dentro do cenário.

In [ ]:
-- 8. DOMINIO DAS CATEGORICAS
(SELECT 'cod_empresa_compensacao' AS coluna, CAST(`cod_empresa_compensacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `cod_empresa_compensacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_tipo_movimento' AS coluna, CAST(`cod_tipo_movimento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `cod_tipo_movimento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_reserva' AS coluna, CAST(`st_reserva` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `st_reserva` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_registro' AS coluna, CAST(`tp_registro` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `tp_registro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_registro_final' AS coluna, CAST(`ind_registro_final` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `ind_registro_final` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_item_eliminado' AS coluna, CAST(`ind_item_eliminado` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `ind_item_eliminado` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_movimento_permitido' AS coluna, CAST(`ind_movimento_permitido` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `ind_movimento_permitido` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_item_dummy' AS coluna, CAST(`ind_item_dummy` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `ind_item_dummy` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_material_granel' AS coluna, CAST(`ind_material_granel` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `ind_material_granel` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_necessidade_atendida' AS coluna, CAST(`ind_necessidade_atendida` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY `ind_necessidade_atendida` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

Campos `double` exigem tolerância de 0,005 na comparação com o SAP.

In [ ]:
-- 9. PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(5,
    'vl_qtd_solicitada', 'decimal(13,3)', COUNT(`vl_qtd_solicitada`), CAST(MIN(`vl_qtd_solicitada`) AS DOUBLE), CAST(MAX(`vl_qtd_solicitada`) AS DOUBLE), CAST(AVG(`vl_qtd_solicitada`) AS DOUBLE), CAST(percentile_approx(`vl_qtd_solicitada`, 0.5) AS DOUBLE), COUNT_IF(`vl_qtd_solicitada` < 0), COUNT_IF(`vl_qtd_solicitada` = 0),
    'vl_qtd_retirada', 'decimal(13,3)', COUNT(`vl_qtd_retirada`), CAST(MIN(`vl_qtd_retirada`) AS DOUBLE), CAST(MAX(`vl_qtd_retirada`) AS DOUBLE), CAST(AVG(`vl_qtd_retirada`) AS DOUBLE), CAST(percentile_approx(`vl_qtd_retirada`, 0.5) AS DOUBLE), COUNT_IF(`vl_qtd_retirada` < 0), COUNT_IF(`vl_qtd_retirada` = 0),
    'qt_estoque_livre_avaliado', 'decimal(13,3)', COUNT(`qt_estoque_livre_avaliado`), CAST(MIN(`qt_estoque_livre_avaliado`) AS DOUBLE), CAST(MAX(`qt_estoque_livre_avaliado`) AS DOUBLE), CAST(AVG(`qt_estoque_livre_avaliado`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_livre_avaliado`, 0.5) AS DOUBLE), COUNT_IF(`qt_estoque_livre_avaliado` < 0), COUNT_IF(`qt_estoque_livre_avaliado` = 0),
    'qt_estoque_consignado', 'decimal(23,3)', COUNT(`qt_estoque_consignado`), CAST(MIN(`qt_estoque_consignado`) AS DOUBLE), CAST(MAX(`qt_estoque_consignado`) AS DOUBLE), CAST(AVG(`qt_estoque_consignado`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_consignado`, 0.5) AS DOUBLE), COUNT_IF(`qt_estoque_consignado` < 0), COUNT_IF(`qt_estoque_consignado` = 0),
    'qt_estoque_projeto', 'decimal(23,3)', COUNT(`qt_estoque_projeto`), CAST(MIN(`qt_estoque_projeto`) AS DOUBLE), CAST(MAX(`qt_estoque_projeto`) AS DOUBLE), CAST(AVG(`qt_estoque_projeto`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_projeto`, 0.5) AS DOUBLE), COUNT_IF(`qt_estoque_projeto` < 0), COUNT_IF(`qt_estoque_projeto` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 10. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel extraído do SAP e compare linha a linha.
Divergência de total é o teste mais rápido para detectar registro faltando ou duplicado —
e cobre o ponto cego da contagem de linhas, que sozinha não prova nada.

In [ ]:
-- 10. TOTAIS PARA CONCILIACAO
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(5,
    'vl_qtd_solicitada', CAST(SUM(`vl_qtd_solicitada`) AS DOUBLE), CAST(ROUND(SUM(`vl_qtd_solicitada`), 2) AS STRING), COUNT(`vl_qtd_solicitada`),
    'vl_qtd_retirada', CAST(SUM(`vl_qtd_retirada`) AS DOUBLE), CAST(ROUND(SUM(`vl_qtd_retirada`), 2) AS STRING), COUNT(`vl_qtd_retirada`),
    'qt_estoque_livre_avaliado', CAST(SUM(`qt_estoque_livre_avaliado`) AS DOUBLE), CAST(ROUND(SUM(`qt_estoque_livre_avaliado`), 2) AS STRING), COUNT(`qt_estoque_livre_avaliado`),
    'qt_estoque_consignado', CAST(SUM(`qt_estoque_consignado`) AS DOUBLE), CAST(ROUND(SUM(`qt_estoque_consignado`), 2) AS STRING), COUNT(`qt_estoque_consignado`),
    'qt_estoque_projeto', CAST(SUM(`qt_estoque_projeto`) AS DOUBLE), CAST(ROUND(SUM(`qt_estoque_projeto`), 2) AS STRING), COUNT(`qt_estoque_projeto`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 11. Datas armazenadas como STRING

**Armadilha:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
Mesma data, formato diferente — normalizar para `AAAAMMDD` antes de comparar.

In [ ]:
-- 11. DATAS EM STRING
SELECT coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo,
       CASE WHEN (CASE WHEN fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato' ELSE 'formato unico' END AS veredito
FROM (
  SELECT stack(1,
    'dt_reserva', COUNT_IF(`dt_reserva` IS NULL OR trim(`dt_reserva`) = ''), COUNT_IF(trim(`dt_reserva`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_reserva`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_reserva`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_reserva`) NOT IN ('', '00000000') THEN `dt_reserva` END), MAX(CASE WHEN trim(`dt_reserva`) NOT IN ('', '00000000') THEN `dt_reserva` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 11.1 Datas em tipo nativo

In [ ]:
-- 11.1 DATAS NATIVAS
SELECT * FROM (
  SELECT stack(1,
    'dt_necessidade', COUNT_IF(`dt_necessidade` IS NULL), CAST(MIN(`dt_necessidade`) AS STRING), CAST(MAX(`dt_necessidade`) AS STRING), COUNT(DISTINCT `dt_necessidade`), COUNT_IF(`dt_necessidade` > current_date())
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 12. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [ ]:
-- 12. CODIGOS
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(5,
    'num_reserva', 'string', COUNT_IF(CAST(`num_reserva` AS STRING) IS NULL OR trim(CAST(`num_reserva` AS STRING)) = ''), MIN(length(trim(CAST(`num_reserva` AS STRING)))), MAX(length(trim(CAST(`num_reserva` AS STRING)))), COUNT_IF(trim(CAST(`num_reserva` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_reserva` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_reserva` AS STRING)), '^0+', '')),
    'num_item_reserva', 'string', COUNT_IF(CAST(`num_item_reserva` AS STRING) IS NULL OR trim(CAST(`num_item_reserva` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_reserva` AS STRING)))), MAX(length(trim(CAST(`num_item_reserva` AS STRING)))), COUNT_IF(trim(CAST(`num_item_reserva` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_reserva` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_reserva` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'num_ordem', 'string', COUNT_IF(CAST(`num_ordem` AS STRING) IS NULL OR trim(CAST(`num_ordem` AS STRING)) = ''), MIN(length(trim(CAST(`num_ordem` AS STRING)))), MAX(length(trim(CAST(`num_ordem` AS STRING)))), COUNT_IF(trim(CAST(`num_ordem` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_ordem` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_ordem` AS STRING)), '^0+', '')),
    'cod_centro_custo', 'string', COUNT_IF(CAST(`cod_centro_custo` AS STRING) IS NULL OR trim(CAST(`cod_centro_custo` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_custo` AS STRING)))), MAX(length(trim(CAST(`cod_centro_custo` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_custo` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro_custo` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_custo` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 13. Chaves normalizadas para join com o SAP

Lista das chaves já **sem zeros à esquerda**, prontas para colar no Excel
e cruzar com o extrato do SAP via PROCV/ÍNDICE.

Baixe como CSV e use para identificar registros presentes de um lado e ausentes do outro.

In [ ]:
-- 13. CHAVES NORMALIZADAS (para cruzar com o SAP)
SELECT DISTINCT
       regexp_replace(trim(CAST(`num_reserva` AS STRING)), '^0+', '') AS `num_reserva_norm`
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
ORDER BY 1;

## 14. Checksum de linha

Gera uma impressão digital de cada linha. Dois usos:

- **Contar linhas realmente distintas** — se `linhas` for maior que `linhas_unicas`,
  existem registros 100% idênticos (duplicata real)
- **Comparação rápida** — aplicando a mesma concatenação no SAP, dá para achar
  divergências sem comparar campo a campo

In [ ]:
-- 14. CHECKSUM DE LINHA
SELECT COUNT(*)                                                 AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_empresa_compensacao` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_produto` AS STRING), ''), COALESCE(CAST(`vl_qtd_solicitada` AS STRING), ''), COALESCE(CAST(`vl_qtd_retirada` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_estoque_livre_avaliado` AS STRING), ''), COALESCE(CAST(`num_item_reserva` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`dt_reserva` AS STRING), ''), COALESCE(CAST(`cod_tipo_movimento` AS STRING), ''), COALESCE(CAST(`dt_necessidade` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`qt_estoque_consignado` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_ponto_descarga` AS STRING), ''), COALESCE(CAST(`nm_recebedor_mercadoria` AS STRING), ''), COALESCE(CAST(`ind_registro_final` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`ind_movimento_permitido` AS STRING), ''), COALESCE(CAST(`num_imobilizado` AS STRING), ''), COALESCE(CAST(`ind_item_dummy` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`num_sub_imobilizado` AS STRING), ''), COALESCE(CAST(`st_reserva` AS STRING), ''), COALESCE(CAST(`tp_registro` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_deposito_destino` AS STRING), ''), COALESCE(CAST(`cod_diagrama_rede` AS STRING), ''), COALESCE(CAST(`num_divisao_programa_venda` AS STRING), ''), COALESCE(CAST(`ind_necessidade_atendida` AS STRING), ''), COALESCE(CAST(`qt_estoque_projeto` AS STRING), ''))))             AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_empresa_compensacao` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_produto` AS STRING), ''), COALESCE(CAST(`vl_qtd_solicitada` AS STRING), ''), COALESCE(CAST(`vl_qtd_retirada` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_estoque_livre_avaliado` AS STRING), ''), COALESCE(CAST(`num_item_reserva` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`dt_reserva` AS STRING), ''), COALESCE(CAST(`cod_tipo_movimento` AS STRING), ''), COALESCE(CAST(`dt_necessidade` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`qt_estoque_consignado` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_ponto_descarga` AS STRING), ''), COALESCE(CAST(`nm_recebedor_mercadoria` AS STRING), ''), COALESCE(CAST(`ind_registro_final` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`ind_movimento_permitido` AS STRING), ''), COALESCE(CAST(`num_imobilizado` AS STRING), ''), COALESCE(CAST(`ind_item_dummy` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`num_sub_imobilizado` AS STRING), ''), COALESCE(CAST(`st_reserva` AS STRING), ''), COALESCE(CAST(`tp_registro` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_deposito_destino` AS STRING), ''), COALESCE(CAST(`cod_diagrama_rede` AS STRING), ''), COALESCE(CAST(`num_divisao_programa_venda` AS STRING), ''), COALESCE(CAST(`ind_necessidade_atendida` AS STRING), ''), COALESCE(CAST(`qt_estoque_projeto` AS STRING), ''))))  AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_empresa_compensacao` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`num_reserva` AS STRING), ''), COALESCE(CAST(`num_ordem` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_produto` AS STRING), ''), COALESCE(CAST(`vl_qtd_solicitada` AS STRING), ''), COALESCE(CAST(`vl_qtd_retirada` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_estoque_livre_avaliado` AS STRING), ''), COALESCE(CAST(`num_item_reserva` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`dt_reserva` AS STRING), ''), COALESCE(CAST(`cod_tipo_movimento` AS STRING), ''), COALESCE(CAST(`dt_necessidade` AS STRING), ''), COALESCE(CAST(`num_lote` AS STRING), ''), COALESCE(CAST(`qt_estoque_consignado` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_ponto_descarga` AS STRING), ''), COALESCE(CAST(`nm_recebedor_mercadoria` AS STRING), ''), COALESCE(CAST(`ind_registro_final` AS STRING), ''), COALESCE(CAST(`ind_item_eliminado` AS STRING), ''), COALESCE(CAST(`ind_movimento_permitido` AS STRING), ''), COALESCE(CAST(`num_imobilizado` AS STRING), ''), COALESCE(CAST(`ind_item_dummy` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`num_sub_imobilizado` AS STRING), ''), COALESCE(CAST(`st_reserva` AS STRING), ''), COALESCE(CAST(`tp_registro` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_deposito_destino` AS STRING), ''), COALESCE(CAST(`cod_diagrama_rede` AS STRING), ''), COALESCE(CAST(`num_divisao_programa_venda` AS STRING), ''), COALESCE(CAST(`ind_necessidade_atendida` AS STRING), ''), COALESCE(CAST(`qt_estoque_projeto` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014';

## 15. Amostra de linhas completas

In [ ]:
-- 15. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
ORDER BY `num_reserva`, `num_item_reserva`
LIMIT 20;

## 16. Distribuição interna do centro 4014

Como o volume se reparte dentro do cenário. Útil para conferir se o extrato do SAP
tem a mesma composição.

In [ ]:
-- 16. DISTRIBUICAO POR cod_tipo_movimento
SELECT COALESCE(NULLIF(trim(CAST(`cod_tipo_movimento` AS STRING)), ''), '(vazio)') AS `cod_tipo_movimento`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_tipo_movimento` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR st_reserva
SELECT COALESCE(NULLIF(trim(CAST(`st_reserva` AS STRING)), ''), '(vazio)') AS `st_reserva`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`st_reserva` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR cod_deposito
SELECT COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)') AS `cod_deposito`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 16. DISTRIBUICAO POR cod_empresa_compensacao
SELECT COALESCE(NULLIF(trim(CAST(`cod_empresa_compensacao` AS STRING)), ''), '(vazio)') AS `cod_empresa_compensacao`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_empresa_compensacao` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [ ]:
-- 17. FRESHNESS
-- Tabela sem coluna de data de ingestao. Use o historico de gravacao.
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_zglmm368 LIMIT 10;

## 18. Análises específicas — ZGLMM368

### 18.1 A chave inclui `num_item_reserva`?

In [ ]:
-- 18.1 ITENS POR RESERVA
SELECT itens_na_reserva, COUNT(*) AS reservas
FROM (SELECT num_reserva, COUNT(DISTINCT num_item_reserva) AS itens_na_reserva
        FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014' GROUP BY num_reserva)
GROUP BY itens_na_reserva
ORDER BY itens_na_reserva
LIMIT 30;

### 18.2 Quantidade solicitada × retirada

In [ ]:
-- 18.2 SOLICITADO x RETIRADO
SELECT COUNT(*) AS linhas,
       COUNT_IF(vl_qtd_retirada > vl_qtd_solicitada) AS retirada_maior,
       COUNT_IF(vl_qtd_retirada = vl_qtd_solicitada) AS totalmente_atendida,
       COUNT_IF(vl_qtd_retirada = 0) AS nada_retirado,
       ROUND(SUM(vl_qtd_solicitada), 3) AS total_solicitado,
       ROUND(SUM(vl_qtd_retirada), 3) AS total_retirado
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014';

### 18.3 Classificação contábil

In [ ]:
-- 18.3 CLASSIFICACAO CONTABIL
SELECT CASE WHEN num_ordem IS NOT NULL AND trim(num_ordem) <> '' THEN 'ordem'
            WHEN cod_centro_custo IS NOT NULL AND trim(cod_centro_custo) <> '' THEN 'centro de custo'
            WHEN num_imobilizado IS NOT NULL AND trim(num_imobilizado) <> '' THEN 'imobilizado'
            ELSE 'sem classificacao' END AS classificacao,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'
GROUP BY 1
ORDER BY linhas DESC;

## 19. EXTRAÇÃO COMPLETA — centro 4014

**Esta é a célula que você baixa para comparar com o SAP.**

Após executar, use **Download → CSV** no resultado.

> **Limites do Databricks:** a tela mostra até 10.000 linhas, mas o download em CSV
> vai além disso. Se o volume for muito grande, use a célula 19.1.

In [ ]:
-- 19. EXTRACAO COMPLETA DO CENARIO
SELECT *
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368
WHERE `cod_centro` = '4014'
ORDER BY `num_reserva`, `num_item_reserva`;

### 19.1 Alternativa para volume grande _(opcional)_

Descomente para gravar o resultado numa tabela própria e exportar de lá sem limite de tela.

In [ ]:
-- 19.1 GRAVAR EXTRACAO EM TABELA (opcional)
-- CREATE OR REPLACE TABLE dev_procurement.corp_curated.extracao_zglmm368_4014 AS
-- SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014';
--
-- SELECT COUNT(*) FROM dev_procurement.corp_curated.extracao_zglmm368_4014;
SELECT 'Descomente as linhas acima se precisar gravar a extracao em tabela' AS instrucao;

## 20. Resumo do cenário

Bloco final. **Copie esta saída** e envie ao agente junto com o notebook.

In [ ]:
-- 20. RESUMO DO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014'),
g AS (
  SELECT 'num_reserva' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_reserva` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_reserva + cod_material' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_reserva`, `cod_material` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_reserva + num_item_reserva' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_reserva + num_item_reserva + cod_material' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `cod_material` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368 WHERE `cod_centro` = '4014')
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'ZGLMM368' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'dev_procurement.corp_curated.tbl_ds_pro_zglmm368'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_centro = 4014'
UNION ALL SELECT 'CENARIO', 'linhas no cenario', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '36'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Baixar a **seção 19** em CSV — é a base do centro 4014 no Datalake.
2. Extrair a mesma transação no SAP com o filtro `centro = 4014`, **todas as abas**.
3. Anotar a data e hora das duas extrações.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Antes de comparar

- [ ] Zeros à esquerda normalizados nos dois lados (seção 12)
- [ ] Formato de data normalizado (seção 11)
- [ ] Totais numéricos conferidos (seção 10)
- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP (seção 6)
